# Trayectorias

Detección y corrección de bosque aislado (clase 3) y plantaciones cortas (9, 35, 74) en MapBiomas Colombia.

`REGION_ID = None` → análisis nacional. Con un ID → región específica.


## Método

1. Detecta bosque aislado (`X-3-Y`, `X-3-3-Y`) y plantación aislada entre bosque.
2. Corrige según el modo configurado (`MODO_CORRECCION`).
3. Analiza y visualiza resultados.

Periodo de detección: 1986–2023. Detalle del método: `metodologia_correccion.txt`.


## 1. Conexión a Earth Engine


In [ ]:
import ee

GEE_PROJECT = 'mapbiomas-colombia'

if 'ee_initialized' not in globals():
    # ee.Authenticate()  # descomentar solo la primera vez
    ee.Initialize(project=GEE_PROJECT)
    ee_initialized = True
    print(f'Conectado a GEE — proyecto: {GEE_PROJECT}')
else:
    print('GEE ya inicializado en esta sesión.')

## 2. Configuración

Definir `REGION_ID`, `MODO_CORRECCION` y `MAPA_ANOMALIAS` antes de continuar.


In [ ]:
import re
import json
from pathlib import Path
import pandas as pd

FOLDER = 'projects/mapbiomas-colombia/assets/LULC/COLECCION4/clasificacion-ft'
REGIONES_XLSX = Path('regiones.xlsx')
REGION_ID = 30450  # None = sin mapa / usa CSV; p. ej. '30453' para mapa
REEXPORTAR = False  # True una vez para regenerar el CSV con la columna year
OUTPUT_CSV = Path('trayectorias_imposibles_por_region.csv')
TOTAL_PIXELES_COLOMBIA_30M = 1_283_824_234  # Colección 3; denominador fijo
YEARS = ee.List.sequence(1986, 2023)
CLASE_BOSQUE = 3
CLASES_PLANTACION = (9, 35, 74)  # silvicultura, palma, plátano/banano

# Matriz de importancia (ATBD Col. 3, Tabla 12). Mayor → menor prevalencia.
PRIORIDAD_CLASES = (
    34,  # 1  Glaciar y nival (cross-cutting; ATBD: sobre todas)
    29,  # 2  Afloramiento rocoso
    32,  # 3  Planicie de marea hipersalina
    30,  # 4  Minería (cross-cutting; ATBD: sobre todas)
    74,  # 5  Plátano y banano (beta)
    75,  # 6  Parques solares
    24,  # 7  Infraestructura urbana
    9,   # 8  Silvicultura
    68,  # 9  Otra área natural sin vegetación
    23,  # 10 Playas, dunas y bancos de arena
    33,  # 11 Río, lago u océano
    35,  # 12 Palma aceitera
    31,  # 13 Acuicultura
    11,  # 17 Formación natural no forestal inundable  [excepción: sobre 21/25]
    5,   # 22 Manglar                               [excepción: sobre 21/25]
    6,   # 23 Bosque inundable                      [excepción: sobre 21/25 y 12/13]
    21,  # 14 Mosaico de agricultura o pasto
    25,  # 15 Otra área sin vegetación
    82,  # 16 Vegetación herbácea/arbustiva andina inundable
    81,  # 19 Vegetación herbácea/arbustiva andina
    13,  # 20 Otra formación natural no forestal
    12,  # 21 Formación herbácea
    3,   # 25 Bosque
    49,  # 26 Vegetación leñosa sobre arena
    50,  # 27 Vegetación herbácea sobre arena
)
VENTANAS_HUECOS = (3, 4, 5)  # A-X-A, A-X-X-A, A-X-X-X-A
CORREGIR_BORDES = False  # evita inventar continuidad fuera del período observado
PASADAS_BOSQUE_RESIDUAL = 2
# 'todas'  → huecos por matriz ATBD + residual bosque↔plantación
# 'bosque' → solo bosque aislado y plantación aislada entre bosque (9/35/74)
MODO_CORRECCION = 'bosque'  # 'todas' | 'bosque'

# Qué pintar en el mapa como trayectorias sospechosas:
# 'bosque'            → solo bosque aislado (clase 3)
# 'bosque_plantacion' → bosque aislado + plantación 9/35/74 aislada entre bosque
MAPA_ANOMALIAS = 'bosque_plantacion'  # 'bosque' | 'bosque_plantacion'

LEYENDA_PATH = Path('leyenda_coleccion3.json')
with open(LEYENDA_PATH, encoding='utf-8') as f:
    _leyenda = json.load(f)
CLASS_NAMES = {int(k): v for k, v in _leyenda['class_names'].items()}
PALETTE_LULC = _leyenda['palette_lulc']

MATRIZ_IMPORTANCIA = pd.DataFrame([
    {
        'rango': i,
        'clase': cid,
        'nombre': CLASS_NAMES.get(cid, f'Clase {cid}'),
        'fuente': 'ATBD Col.3 Tabla 12' + (
            ' + excepción inundable' if cid in (5, 6, 11) else ''
        ),
    }
    for i, cid in enumerate(PRIORIDAD_CLASES, start=1)
])
print(f'Matriz de importancia: {len(PRIORIDAD_CLASES)} clases de nivel 2')
print(f"Mapa anomalías: {MAPA_ANOMALIAS}")
display(MATRIZ_IMPORTANCIA)

VIS_ANOMALIAS = {'min': 1, 'max': 4, 'palette': ['#fee08b', '#fdae61', '#f46d43', '#a50026']}
LEGEND_ANOMALIAS = {
    'Bosque 1': 'fee08b',
    'Bosque 2': 'fdae61',
    'Bosque 3': 'f46d43',
    'Bosque 4+': 'a50026',
}
VIS_ANOMALIAS_PLANT = {'min': 1, 'max': 4, 'palette': ['#a5d8ff', '#4dabf7', '#228be6', '#1864ab']}
LEGEND_ANOMALIAS_PLANT = {
    'Plantacion 1': 'a5d8ff',
    'Plantacion 2': '4dabf7',
    'Plantacion 3': '228be6',
    'Plantacion 4+': '1864ab',
}
LEGEND_ANOMALIAS_MIXTO = {**LEGEND_ANOMALIAS, **LEGEND_ANOMALIAS_PLANT}
VIS_LULC = {'min': 0, 'max': len(PALETTE_LULC) - 1, 'palette': PALETTE_LULC}


## 3. Funciones

Detección, corrección y utilidades de Earth Engine.


In [ ]:

def class_name(class_id):
    return CLASS_NAMES.get(int(class_id), f'Clase {int(class_id)}')


def lulc_color(class_id):
    idx = int(class_id)
    return PALETTE_LULC[idx] if 0 <= idx < len(PALETTE_LULC) else '#888888'


def band_name(year):
    return ee.String('classification_').cat(ee.Number(year).format('%d'))


def es_plantacion(img):
    """Silvicultura (9), palma (35) o plátano/banano (74)."""
    mask = img.eq(CLASES_PLANTACION[0])
    for cid in CLASES_PLANTACION[1:]:
        mask = mask.Or(img.eq(cid))
    return mask


def reemplazo_falso_bosque(prev, next1):
    """Bosque aislado 1 año (o 2º año del bloque): plantación solo en t+1; si no, t−1."""
    rep = ee.Image(prev)
    for cid in CLASES_PLANTACION:
        rep = rep.where(next1.eq(cid), cid)
    return rep


def reemplazo_falso_bosque_2anio_ini(prev, next1, next2):
    """Primer 3 de bloque de 2 años: si t+1 es otro bosque y t+2 es plantación, usar t+2."""
    rep = ee.Image(prev)
    for cid in CLASES_PLANTACION:
        rep = rep.where(next1.eq(cid), cid)
    seguido_de_bosque = next1.eq(CLASE_BOSQUE)
    for cid in CLASES_PLANTACION:
        rep = rep.where(seguido_de_bosque.And(next2.eq(cid)), cid)
    return rep


def anomalia_bosque(prev1, curr, next1, next2):
    """Misma detección que corregir_falsos_bosques: bosque aislado 1 o 2 años."""
    error1 = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.neq(CLASE_BOSQUE))
    error2_ini = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.eq(CLASE_BOSQUE)).And(next2.neq(CLASE_BOSQUE))
    return error1.Or(error2_ini)


def anomalia_plantacion(prev1, curr, next1, next2):
    """Plantación aislada 1–2 años entre bosque (9/35/74)."""
    error1 = prev1.eq(CLASE_BOSQUE).And(es_plantacion(curr)).And(next1.eq(CLASE_BOSQUE))
    error2_ini = (
        prev1.eq(CLASE_BOSQUE)
        .And(es_plantacion(curr))
        .And(es_plantacion(next1))
        .And(next2.eq(CLASE_BOSQUE))
    )
    return error1.Or(error2_ini)


def anomalia_mapa(prev1, curr, next1, next2, modo='bosque'):
    """Detección para capas del mapa según selector."""
    modo = str(modo or 'bosque').strip().lower()
    a = ee.Image(anomalia_bosque(prev1, curr, next1, next2)).unmask(0)
    if modo in ('bosque_plantacion', 'bosque+plantacion') or 'plant' in modo:
        a = a.Or(ee.Image(anomalia_plantacion(prev1, curr, next1, next2)).unmask(0))
    return a.gt(0)


def describe_trayectoria(p1, c, n1, n2):
    p1, c, n1, n2 = map(int, (p1, c, n1, n2))
    plant = set(CLASES_PLANTACION)
    if c == CLASE_BOSQUE:
        if n1 != CLASE_BOSQUE and n2 != CLASE_BOSQUE:
            return 'Bosque aislado 1 año'
        if n1 == CLASE_BOSQUE and n2 != CLASE_BOSQUE:
            return 'Bosque aislado 2 años'
        return 'Revisar patrón'
    if c in plant:
        if p1 == CLASE_BOSQUE and n1 == CLASE_BOSQUE:
            return 'Plantación aislada 1 año'
        if p1 == CLASE_BOSQUE and n1 in plant and n2 == CLASE_BOSQUE:
            return 'Plantación aislada 2 años'
        return 'Revisar plantación'
    return 'Otro patrón'


def codigo_clases(p1, c, n1, n2):
    p1, c, n1, n2 = map(int, (p1, c, n1, n2))
    return f'{p1}-{c}-{n1}-{n2}'


def codigo_desde_k(k):
    k = int(float(k))
    return codigo_clases(
        k // 1_000_000,
        (k % 1_000_000) // 10_000,
        (k % 10_000) // 100,
        k % 100,
    )


def parse_trajectory(code):
    k = int(float(code))
    p1, c, n1, n2 = k // 1_000_000, (k % 1_000_000) // 10_000, (k % 10_000) // 100, k % 100
    ids = codigo_clases(p1, c, n1, n2)
    return {
        'codigo': k,
        'clase_1': p1, 'clase_2': c, 'clase_3': n1, 'clase_4': n2,
        'clases': ids,
        'trayectoria': ids,
        'tipo': describe_trayectoria(p1, c, n1, n2),
    }


def cargar_asset_region(region_id, folder=FOLDER):
    """Busca COLOMBIA-{region_id}-{version} en GEE; region_id viene del Excel."""
    rid = str(region_id).strip()
    prefix = f'COLOMBIA-{rid}-'
    assets = ee.data.listAssets({'parent': folder})['assets']
    matches = []
    for a in assets:
        base = a['name'].split('/')[-1]
        if not base.startswith(prefix):
            continue
        ver = base[len(prefix):]
        if ver.isdigit() and int(ver) != 11 and int(ver) != 99:
            matches.append(a)
    matches = sorted(matches, key=lambda a: int(a['name'].split('/')[-1].rsplit('-', 1)[-1]), reverse=True)
    if not matches:
        return None, None
    name = matches[0]['name']
    return ee.Image(name), name


def cargar_mosaico_regiones(region_ids, folder=FOLDER):
    """Mosaico con la versión más reciente de todas las regiones disponibles."""
    ids = {str(rid).strip() for rid in region_ids}
    disponibles = {}
    for asset in ee.data.listAssets({'parent': folder})['assets']:
        name = asset['name']
        base = name.split('/')[-1]
        if not base.startswith('COLOMBIA-'):
            continue
        body = base[len('COLOMBIA-'):]
        if '-' not in body:
            continue
        rid, version = body.rsplit('-', 1)
        if rid not in ids or not version.isdigit() or int(version) in (11, 99):
            continue
        disponibles.setdefault(rid, []).append((int(version), name))

    seleccionados = {
        rid: max(opciones, key=lambda item: item[0])[1]
        for rid, opciones in disponibles.items()
    }
    nombres = [seleccionados[str(rid)] for rid in region_ids if str(rid) in seleccionados]
    faltantes = [str(rid) for rid in region_ids if str(rid) not in seleccionados]
    bandas = [f'classification_{year}' for year in range(1985, 2026)]
    imagenes = [ee.Image(name).select(bandas).toByte() for name in nombres]
    mosaico = ee.ImageCollection.fromImages(imagenes).mosaic()
    return mosaico, nombres, faltantes


def histogramas_trayectorias_por_anio(imagen):
    """Histogramas de trayectorias por año en una sola reducción regional."""
    bandas = []
    for year in range(1986, 2024):
        prev1 = imagen.select(band_name(year - 1))
        curr = imagen.select(band_name(year))
        next1 = imagen.select(band_name(year + 1))
        next2 = imagen.select(band_name(year + 2))
        anomalia = anomalia_bosque(prev1, curr, next1, next2)
        trayectoria = prev1.multiply(1_000_000).add(curr.multiply(10_000)).add(next1.multiply(100)).add(next2)
        bandas.append(trayectoria.updateMask(anomalia).rename(f'trajectory_{year}'))

    hist = ee.Image.cat(bandas).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=imagen.geometry(), scale=30, maxPixels=1e10, tileScale=8,
    ).getInfo()
    return {
        year: hist.get(f'trajectory_{year}') or {}
        for year in range(1986, 2024)
    }


def ids_desde_regiones(path='regiones.xlsx'):
    """Lee IDs de región desde la primera columna del xlsx (sin encabezado), en orden del archivo."""
    col = pd.read_excel(path, header=None, usecols=[0]).iloc[:, 0].dropna()
    ids = col.astype(int).astype(str).drop_duplicates().tolist()
    if not ids:
        raise ValueError(f'No se encontraron IDs de región en {path}')
    return ids


def corregir_falsos_bosques(imagen, pasadas=2):
    """Corrige en bloque bosque aislado residual, incluido A-3-B."""

    def _paso(img):
        # Años centrales con ventana completa t-2 ... t+2.
        years = list(range(1987, 2024))
        curr_names = [f'classification_{y}' for y in years]

        def serie(offset):
            names = [f'classification_{y + offset}' for y in years]
            return img.select(names).rename(curr_names)

        prev2, prev1 = serie(-2), serie(-1)
        curr, next1, next2 = serie(0), serie(1), serie(2)

        error1 = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.neq(CLASE_BOSQUE))
        error2_ini = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.eq(CLASE_BOSQUE)).And(next2.neq(CLASE_BOSQUE))
        error2_fin = prev2.neq(CLASE_BOSQUE).And(prev1.eq(CLASE_BOSQUE)).And(curr.eq(CLASE_BOSQUE)).And(next1.neq(CLASE_BOSQUE))

        rep1 = reemplazo_falso_bosque(prev1, next1)
        rep2_ini = reemplazo_falso_bosque_2anio_ini(prev1, next1, next2)
        rep2_fin = reemplazo_falso_bosque(prev2, next1)
        corrected = curr.where(error1, rep1).where(error2_ini, rep2_ini).where(error2_fin, rep2_fin)
        out = img.addBands(corrected, curr_names, True)

        # Bordes sin ventana de cinco años: solo A-3-B.
        for y in (1986, 2024):
            curr_edge = out.select(band_name(y))
            prev_edge = out.select(band_name(y - 1))
            next_edge = out.select(band_name(y + 1))
            error = prev_edge.neq(CLASE_BOSQUE).And(curr_edge.eq(CLASE_BOSQUE)).And(next_edge.neq(CLASE_BOSQUE))
            fixed = curr_edge.where(error, reemplazo_falso_bosque(prev_edge, next_edge))
            out = out.addBands(fixed, [f'classification_{y}'], True)
        return out.toByte()

    result = imagen
    for _ in range(pasadas):
        result = _paso(result)
    return result


def _rellenar_borde_temporal(imagen, valor, year_min=1985, year_max=2025):
    """Extiende una clase estable de dos años hacia el primer/último año."""
    first = imagen.select(band_name(year_min))
    second = imagen.select(band_name(year_min + 1))
    third = imagen.select(band_name(year_min + 2))
    mask_first = first.neq(valor).And(second.eq(valor)).And(third.eq(valor))
    imagen = imagen.addBands(first.where(mask_first, valor), [f'classification_{year_min}'], True)

    before2 = imagen.select(band_name(year_max - 2))
    before1 = imagen.select(band_name(year_max - 1))
    last = imagen.select(band_name(year_max))
    mask_last = before2.eq(valor).And(before1.eq(valor)).And(last.neq(valor))
    return imagen.addBands(last.where(mask_last, valor), [f'classification_{year_max}'], True)


def _rellenar_hueco(imagen, valor, longitud, year_min=1985, year_max=2025):
    """Rellena A-X...-A; longitud 1, 2 o 3 años internos."""
    starts = list(range(year_min + 1, year_max - longitud + 1))
    prev_names = [f'classification_{y - 1}' for y in starts]
    after_names = [f'classification_{y + longitud}' for y in starts]
    prev = imagen.select(prev_names)
    after = imagen.select(after_names)
    mask = prev.eq(valor).And(after.eq(valor))

    for offset in range(longitud):
        names = [f'classification_{y + offset}' for y in starts]
        inside = imagen.select(names)
        mask = mask.And(inside.neq(valor))

    for offset in range(longitud):
        names = [f'classification_{y + offset}' for y in starts]
        inside = imagen.select(names)
        corrected = inside.where(mask.rename(inside.bandNames()), valor)
        imagen = imagen.addBands(corrected, names, True)
    return imagen


def corregir_plantaciones_aisladas(imagen, pasadas=2):
    """Plantación/cultivo aislado 1–2 años entre bosque → bosque.

    Ej.: 3-3-3-9-9-3-3-3 → todo 3. Complementa corregir_falsos_bosques
    (9-9-9-3-3-9-9-9 → todo 9).
    """

    def _paso(img):
        years = list(range(1987, 2024))
        curr_names = [f'classification_{y}' for y in years]

        def serie(offset):
            names = [f'classification_{y + offset}' for y in years]
            return img.select(names).rename(curr_names)

        prev2, prev1 = serie(-2), serie(-1)
        curr, next1, next2 = serie(0), serie(1), serie(2)

        error1 = prev1.eq(CLASE_BOSQUE).And(es_plantacion(curr)).And(next1.eq(CLASE_BOSQUE))
        error2_ini = (
            prev1.eq(CLASE_BOSQUE)
            .And(es_plantacion(curr))
            .And(es_plantacion(next1))
            .And(next2.eq(CLASE_BOSQUE))
        )
        error2_fin = (
            prev2.eq(CLASE_BOSQUE)
            .And(es_plantacion(prev1))
            .And(es_plantacion(curr))
            .And(next1.eq(CLASE_BOSQUE))
        )

        corrected = (
            curr.where(error1, CLASE_BOSQUE)
            .where(error2_ini, CLASE_BOSQUE)
            .where(error2_fin, CLASE_BOSQUE)
        )
        out = img.addBands(corrected, curr_names, True)

        for y in (1986, 2024):
            curr_edge = out.select(band_name(y))
            prev_edge = out.select(band_name(y - 1))
            next_edge = out.select(band_name(y + 1))
            error = (
                prev_edge.eq(CLASE_BOSQUE)
                .And(es_plantacion(curr_edge))
                .And(next_edge.eq(CLASE_BOSQUE))
            )
            fixed = curr_edge.where(error, CLASE_BOSQUE)
            out = out.addBands(fixed, [f'classification_{y}'], True)
        return out.toByte()

    result = imagen
    for _ in range(pasadas):
        result = _paso(result)
    return result


def corregir_bosque_y_plantaciones(imagen, pasadas=2):
    """Alterna falsos bosques y plantaciones aisladas entre bosque."""
    result = imagen
    for _ in range(pasadas):
        result = corregir_falsos_bosques(result, pasadas=1)
        result = corregir_plantaciones_aisladas(result, pasadas=1)
    return result


def corregir_matriz_importancia(
    imagen,
    prioridad=PRIORIDAD_CLASES,
    ventanas=VENTANAS_HUECOS,
    corregir_bordes=CORREGIR_BORDES,
    pasadas_bosque=PASADAS_BOSQUE_RESIDUAL,
    modo=None,
):
    """Corrección unificada según MODO_CORRECCION.

    - 'todas': huecos seguros por matriz de prioridad + residual bosque↔plantación.
    - 'bosque': bosque aislado y plantación aislada entre bosque (9/35/74).

    `prioridad` va de mayor a menor. Se ejecuta al revés para que la clase de
    mayor prioridad sea la última en resolver cualquier cascada temporal.
    """
    modo = (modo or MODO_CORRECCION or 'todas').strip().lower()
    if modo not in ('todas', 'bosque'):
        raise ValueError("MODO_CORRECCION debe ser 'todas' o 'bosque'")

    if modo == 'bosque':
        result = corregir_bosque_y_plantaciones(imagen, pasadas=pasadas_bosque)
        return result.select(imagen.bandNames()).toByte()

    result = imagen.toByte()
    orden_ejecucion = list(reversed(tuple(prioridad)))

    if corregir_bordes:
        for valor in orden_ejecucion:
            result = _rellenar_borde_temporal(result, valor)

    # Ventana corta, ventanas largas y una repetición corta para residuales.
    if 3 in ventanas:
        for valor in orden_ejecucion:
            result = _rellenar_hueco(result, valor, longitud=1)
    for ventana in (4, 5):
        if ventana in ventanas:
            for valor in orden_ejecucion:
                result = _rellenar_hueco(result, valor, longitud=ventana - 2)
    if 3 in ventanas:
        for valor in orden_ejecucion:
            result = _rellenar_hueco(result, valor, longitud=1)

    result = corregir_bosque_y_plantaciones(result, pasadas=pasadas_bosque)
    return result.select(imagen.bandNames()).toByte()


## 4. Carga de datos

Ejecutar de nuevo tras cambiar `REGION_ID` o el modo de corrección.


In [ ]:

if not REGIONES_XLSX.exists():
    raise FileNotFoundError(f'Coloca {REGIONES_XLSX} en la carpeta del proyecto')

REGION_IDS = ids_desde_regiones(REGIONES_XLSX)
MODO_MAPA = REGION_ID is not None


def _msg_correccion():
    modo = str(MODO_CORRECCION).strip().lower()
    plant = '/'.join(str(c) for c in CLASES_PLANTACION)
    if modo == 'bosque':
        return (
            f"Corrección: bosque↔plantación ({plant}) "
            f"· {PASADAS_BOSQUE_RESIDUAL} pasadas"
        )
    return (
        f"Corrección: matriz ATBD ({len(PRIORIDAD_CLASES)} clases) "
        f"+ residual bosque↔plantación ({plant}) · modo '{modo}'"
    )


print(f'Leyenda: {len(CLASS_NAMES)} clases desde {_leyenda["source"]}')
print(f'Regiones en {REGIONES_XLSX.name}: {len(REGION_IDS)}')
print(f'Modo corrección: {MODO_CORRECCION}')

if MODO_MAPA:
    rid = str(REGION_ID)
    if rid not in REGION_IDS:
        print(f'Aviso: {rid} no aparece en {REGIONES_XLSX.name}')
    IMG_ORIGINAL, ASSET_ORIGINAL = cargar_asset_region(rid)
    if IMG_ORIGINAL is None:
        raise ValueError(f'No hay asset LULC para la región {rid}')
    IMG_CORREGIDA = corregir_matriz_importancia(IMG_ORIGINAL)
    print(f'Región cargada: {rid}')
    print(f'Asset: {ASSET_ORIGINAL}')
    print(_msg_correccion())
else:
    IMG_ORIGINAL, ASSETS_MOSAICO, REGIONES_FALTANTES = cargar_mosaico_regiones(REGION_IDS)
    IMG_CORREGIDA = corregir_matriz_importancia(IMG_ORIGINAL)
    ASSET_ORIGINAL = 'MOSAICO_REGIONES'
    print(f'Mosaico nacional cargado: {len(ASSETS_MOSAICO)}/{len(REGION_IDS)} regiones con asset')
    print(_msg_correccion())
    if REGIONES_FALTANTES:
        print(f'Regiones sin asset: {len(REGIONES_FALTANTES)}')


## 5. Exportar por regiones


In [ ]:
if MODO_MAPA:
    print('Modo mapa activo: export batch omitido.')
elif not REEXPORTAR and OUTPUT_CSV.exists():
    print(f'CSV ya existe → no se reexporta: {OUTPUT_CSV.resolve()}')
    print('Para regenerarlo, pon REEXPORTAR = True en Configuración y vuelve a ejecutar.')
    df_tray = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    print(f'Filas: {len(df_tray):,} · Regiones: {df_tray["region_id"].nunique()}')
    display(df_tray.head(10))
elif not REEXPORTAR and not OUTPUT_CSV.exists():
    raise FileNotFoundError(
        f'No existe {OUTPUT_CSV}. Pon REEXPORTAR = True para generarlo, '
        'o coloca el CSV en la carpeta del proyecto.'
    )
else:
    region_ids = REGION_IDS
    print(f'Reexportando {len(region_ids)} regiones → {OUTPUT_CSV.name}')

    filas = []
    omitidas = []
    sin_anomalias = []

    for i, rid in enumerate(region_ids, 1):
        print(f'[{i}/{len(region_ids)}] {rid}...', end=' ')
        try:
            img, asset = cargar_asset_region(rid)
        except Exception as err:
            print(f'error ({err})')
            omitidas.append({'region_id': rid, 'motivo': str(err)})
            continue
        if img is None:
            print('sin asset')
            omitidas.append({'region_id': rid, 'motivo': 'sin asset'})
            continue
        try:
            hist_por_anio = histogramas_trayectorias_por_anio(img)
        except Exception as err:
            print(f'error GEE ({err})')
            omitidas.append({'region_id': rid, 'motivo': str(err), 'asset': asset})
            continue
        if not any(hist_por_anio.values()):
            print('sin anomalias')
            sin_anomalias.append(rid)
            continue
        n_patrones = 0
        for year, hist in hist_por_anio.items():
            for codigo, pixeles in hist.items():
                filas.append({
                    'region_id': rid,
                    'year': year,
                    'asset': asset,
                    **parse_trajectory(codigo),
                    'pixeles': int(pixeles),
                })
                n_patrones += 1
        print(f'{n_patrones} patrones-año')

    df_tray = pd.DataFrame(filas).sort_values(
        ['region_id', 'year', 'pixeles'], ascending=[True, True, False]
    )
    df_tray.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f'\nGuardado: {OUTPUT_CSV.resolve()}')
    n_reg = df_tray['region_id'].nunique() if len(df_tray) else 0
    print(f'Filas: {len(df_tray):,} · Regiones con datos: {n_reg}')
    print(f'Omitidas: {len(omitidas)} · Sin anomalias: {len(sin_anomalias)}')
    if omitidas:
        display(pd.DataFrame(omitidas).head(10))
    display(df_tray.head(10))


## 6. Análisis nacional o regional

`REGION_ID = None` → nacional. Con valor → regional.


In [ ]:
from importlib import reload
import matplotlib.pyplot as plt
import analisis_nacional

reload(analisis_nacional)
plt.style.use('default')
RESULTADOS = analisis_nacional.analizar(
    region_id=REGION_ID,
    total_colombia=TOTAL_PIXELES_COLOMBIA_30M,
)
print('Listo. Claves:', list(RESULTADOS.keys()))


## 6b. Final de la serie (2023–2025)

Compara bloques de bosque de 1 año y de 2 años, y el año en que finalizan.


In [ ]:
from importlib import reload
import analisis_extremos

reload(analisis_extremos)
EXTREMOS = analisis_extremos.analizar_extremos()
print('Claves:', list(EXTREMOS.keys()))
EXTREMOS['resumen_2023']

## 7. Presentación PPTX


In [ ]:
from importlib import reload
import export_presentacion

reload(export_presentacion)
export_presentacion.main()


## 8. Mapa interactivo

Clic en el mapa para marcar un píxel y comparar la serie original con la corregida.


In [ ]:
if IMG_ORIGINAL is None:
    from IPython.display import HTML, clear_output, display
    clear_output(wait=True)
    display(HTML(
        '<p style="font-family:system-ui;color:#666">'
        'Ejecuta primero la celda de carga de datos.</p>'
    ))
else:
    import io
    import time
    import ipywidgets as widgets
    import leafmap
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from IPython.display import clear_output, display

    # Estilo claro para el gráfico del píxel.
    plt.style.use('default')
    plt.rcParams.update({
        'text.color': '#1a202c',
        'axes.labelcolor': '#1a202c',
        'axes.titlecolor': '#1a202c',
        'xtick.color': '#4a5568',
        'ytick.color': '#4a5568',
        'legend.labelcolor': '#1a202c',
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
    })

    if '_MAP_UI' not in globals():
        _MAP_UI = {'seq': 0, 'busy': False}
    _MAP_UI['seq'] += 1
    _MAP_UI['busy'] = False
    MAP_SEQ = _MAP_UI['seq']

    state = {
        'current_img': IMG_ORIGINAL,
        'trajectories_img': None,
        'layers': [],
        'fase': 'Original',
        'last_click_ts': 0.0,
        'map_seq': MAP_SEQ,
        'legend_added': False,
        'pixel': None,  # (lat, lon)
        'marker': None,
        'vista_inicializada': False,  # fit_bounds solo la primera vez
    }

    fase_label = widgets.HTML(value='<span style="color:#4a5568">Vista activa: <b>Original</b></span>')
    chart_title = widgets.HTML(
        value="<div style='color:#718096;font-style:italic;padding:48px 12px;text-align:center'>"
              "Clic en el mapa para marcar un píxel y ver original vs corregido.</div>"
    )
    chart_img = widgets.Image(format='png', layout=widgets.Layout(width='100%', display='none'))
    chart_box = widgets.VBox(
        [chart_title, chart_img],
        layout=widgets.Layout(
            width='100%', min_height='480px', padding='12px',
            border='1px solid #cbd5e0', border_radius='10px',
        ),
    )
    table_html = widgets.HTML(
        value="<div style='color:#718096;padding:16px'>Calculando trayectorias…</div>",
        layout=widgets.Layout(
            width='100%', min_height='320px', max_height='520px', overflow_y='auto',
            padding='8px', border='1px solid #cbd5e0', border_radius='10px',
        ),
    )

    def _section_title(title, subtitle=''):
        sub = f'<div style="font-size:12px;color:#718096;margin-top:4px">{subtitle}</div>' if subtitle else ''
        return widgets.HTML(
            f'<div style="font-family:system-ui;width:100%;border-bottom:2px solid #2b6cb0;'
            f'padding:10px 0 8px 0;margin:20px 0 12px 0">'
            f'<div style="font-size:17px;font-weight:700;color:#1a365d">{title}</div>{sub}</div>'
        )

    m = leafmap.Map(center=[4.5, -73.0], zoom=6, height='680px')
    m.layout = widgets.Layout(width='100%', height='680px', margin='0 0 8px 0')
    m.add_basemap('CartoDB.DarkMatter')
    m.add_layer_control(position='topright')

    TABLE_CSS = """
    <style>
    .traj-wrap { font-family: system-ui, sans-serif; }
    .traj-wrap h3 { margin: 0 0 12px 0; font-size: 15px; color: #1a202c; }
    .traj-wrap .meta { color: #718096; font-size: 12px; margin-bottom: 10px; }
    .traj-table { width: 100%; border-collapse: collapse; font-size: 12px; }
    .traj-table th {
        background: #2d3748; color: #fff; padding: 10px 8px; text-align: left;
        position: sticky; top: 0; z-index: 1;
    }
    .traj-table td { padding: 8px; border-bottom: 1px solid #e2e8f0; vertical-align: top; }
    .traj-table td.traj { max-width: 480px; line-height: 1.4; word-wrap: break-word; }
    .traj-table tr:nth-child(even) { background: #f8fafc; }
    .traj-table tr:hover { background: #edf2f7; }
    .traj-table .num { text-align: right; white-space: nowrap; font-variant-numeric: tabular-nums; }
    .empty { color: #718096; font-style: italic; padding: 24px 8px; }
    </style>
    """


    def mostrar_placeholder_pixel(msg='Clic en el mapa para marcar un píxel y ver original vs corregido.'):
        chart_img.layout.display = 'none'
        chart_img.value = b''
        chart_title.value = (
            f"<div style='color:#718096;font-style:italic;padding:48px 12px;text-align:center'>{msg}</div>"
        )


    def mostrar_pixel_cargando(lat, lon, texto='Consultando serie original y corregida…'):
        chart_img.layout.display = 'none'
        chart_img.value = b''
        chart_title.value = (
            f"<div style='font-family:system-ui;color:#1a202c;padding:8px 4px'>"
            f"<b>Píxel marcado</b> · {lat:.4f}, {lon:.4f}<br>"
            f"<span style='color:#718096;font-size:12px'>{texto}</span></div>"
        )


    def mostrar_pixel_resultado(lat, lon, png_bytes=None, empty=False, error=None, n_cambios=None):
        if error:
            chart_img.layout.display = 'none'
            chart_img.value = b''
            chart_title.value = (
                f"<div style='font-family:system-ui;color:#c53030;padding:12px'>"
                f"Error en píxel {lat:.4f}, {lon:.4f}: {error}</div>"
            )
            return
        if empty or not png_bytes:
            chart_img.layout.display = 'none'
            chart_img.value = b''
            chart_title.value = (
                f"<div style='font-family:system-ui;color:#718096;padding:12px'>"
                f"Píxel {lat:.4f}, {lon:.4f}: sin datos.</div>"
            )
            return
        extra = ''
        if n_cambios is not None:
            if n_cambios == 0:
                extra = " · <span style='color:#2f855a'>sin cambios tras la corrección</span>"
            else:
                extra = f" · <span style='color:#c05621'><b>{n_cambios} año(s) cambiaron</b></span>"
        chart_title.value = (
            f"<div style='font-family:system-ui;color:#1a202c;padding:4px 0 8px 0'>"
            f"<b>Píxel marcado</b> · {lat:.4f}, {lon:.4f}{extra}<br>"
            f"<span style='color:#718096;font-size:12px'>"
            f"Arriba: original · Abajo: corregida · Puntos rojos = años distintos</span></div>"
        )
        chart_img.value = png_bytes
        chart_img.layout.display = 'block'


    def _capturar_vista():
        try:
            return {'center': list(m.center), 'zoom': float(m.zoom)}
        except Exception:
            return None


    def _restaurar_vista(vista):
        if not vista:
            return
        try:
            m.center = vista['center']
            m.zoom = vista['zoom']
        except Exception:
            pass


    def _restaurar_vista_diferida(vista, delay=0.2):
        """ipyleaflet a veces resetea el zoom tras remount del layout; reaplicar."""
        _restaurar_vista(vista)
        if not vista:
            return

        def _again():
            _restaurar_vista(vista)

        try:
            import threading
            threading.Timer(delay, _again).start()
        except Exception:
            pass


    def poner_marcador(lat, lon):
        """Mueve el marcador existente o lo crea; no altera centro/zoom."""
        vista = _capturar_vista()
        marker = state.get('marker')
        if marker is not None:
            try:
                marker.location = (lat, lon)
                _restaurar_vista_diferida(vista)
                return
            except Exception:
                try:
                    m.remove_layer(marker)
                except Exception:
                    pass
                state['marker'] = None

        old = m.find_layer('Punto seleccionado')
        if old is not None:
            m.remove_layer(old)
        m.add_marker(location=[lat, lon], name='Punto seleccionado', draggable=False)
        state['marker'] = m.find_layer('Punto seleccionado')
        _restaurar_vista_diferida(vista)


    def quitar_capas_analisis():
        """Quita capas LULC/anomalías, pero conserva el marcador del píxel."""
        for name in state['layers']:
            layer = m.find_layer(name)
            if layer is not None:
                m.remove_layer(layer)
        state['layers'] = []


    def html_tabla_trayectorias(df, fase):
        total_pix = int(df['pixeles'].sum())
        show = df.head(15).copy()
        rows = []
        for _, r in show.iterrows():
            rows.append(
                f"<tr><td class='traj'>{r['trayectoria']}</td>"
                f"<td>{r['tipo']}</td>"
                f"<td class='num'>{int(r['pixeles']):,}</td></tr>"
            )
        body = ''.join(rows) if rows else "<tr><td colspan='3' class='empty'>Sin anomalías detectadas.</td></tr>"
        return (
            f"{TABLE_CSS}<div class='traj-wrap'>"
            f"<h3>Trayectorias imposibles — {fase}</h3>"
            f"<div class='meta'>{len(df):,} patrones · {total_pix:,} píxeles anómalos · top 15</div>"
            f"<table class='traj-table'><thead><tr>"
            f"<th>ID trayectoria</th><th>Tipo</th><th>Píxeles</th>"
            f"</tr></thead><tbody>{body}</tbody></table></div>"
        )


    def props_a_serie(properties):
        df = pd.DataFrame(
            [(int(k), int(v)) for k, v in properties.items() if str(k).isdigit() and v is not None],
            columns=['Año', 'Clase'],
        ).sort_values('Año')
        return df


    def sample_pixel_serie(img, lon, lat):
        point = ee.Geometry.Point([lon, lat])
        bands = img.bandNames()
        years = bands.map(lambda b: ee.String(ee.List(ee.String(b).split('_')).get(1)))
        return img.select(bands, years).reduceRegion(
            reducer=ee.Reducer.first(), geometry=point, scale=30,
        ).getInfo()


    def _dibujar_serie(ax, df, titulo, highlight_years=None):
        highlight_years = set(highlight_years or [])
        colors = df['Clase'].map(lulc_color)
        ax.set_facecolor('#f7fafc')
        ax.axhspan(CLASE_BOSQUE - 0.4, CLASE_BOSQUE + 0.4, color='#1f8d49', alpha=0.12)
        ax.plot(df['Año'], df['Clase'], color='#a0aec0', linestyle='-', alpha=0.7, zorder=1)
        ax.scatter(df['Año'], df['Clase'], c=colors, s=70, edgecolors='#2d3748', linewidths=0.5, zorder=2)
        if highlight_years:
            mask = df['Año'].isin(highlight_years)
            ax.scatter(
                df.loc[mask, 'Año'], df.loc[mask, 'Clase'],
                s=160, facecolors='none', edgecolors='#c53030', linewidths=2.2, zorder=3,
                label='Año corregido',
            )
        ax.set_ylim(0, min(len(PALETTE_LULC) - 1, max(40, int(df['Clase'].max()) + 2)))
        ax.set_title(titulo, fontsize=11, color='#1a202c', loc='left')
        ax.set_ylabel('Clase LULC', color='#1a202c')
        ax.tick_params(colors='#4a5568', labelsize=8)
        ax.grid(axis='y', color='#e2e8f0')
        return sorted(df['Clase'].unique())


    def render_compare_png(lat, lon, df_orig, df_corr):
        if df_orig.empty and df_corr.empty:
            return None, 0

        merged = df_orig.merge(df_corr, on='Año', how='outer', suffixes=('_orig', '_corr')).sort_values('Año')
        cambiados = merged[
            merged['Clase_orig'].notna() & merged['Clase_corr'].notna()
            & (merged['Clase_orig'] != merged['Clase_corr'])
        ]['Año'].astype(int).tolist()

        fig, axes = plt.subplots(2, 1, figsize=(12, 6.4), sharex=True, facecolor='white')
        clases = set()
        if not df_orig.empty:
            clases.update(_dibujar_serie(axes[0], df_orig, 'Original', highlight_years=cambiados))
        else:
            axes[0].text(0.5, 0.5, 'Sin datos original', ha='center', va='center', transform=axes[0].transAxes)
        if not df_corr.empty:
            clases.update(_dibujar_serie(axes[1], df_corr, 'Corregida (matriz)', highlight_years=cambiados))
        else:
            axes[1].text(0.5, 0.5, 'Sin datos corregida', ha='center', va='center', transform=axes[1].transAxes)

        axes[1].set_xlabel('Año', color='#1a202c')
        years = sorted(set(df_orig['Año']).union(df_corr['Año']))
        if years:
            step = 2 if len(years) > 20 else 1
            ticks = years[::step]
            axes[1].set_xticks(ticks)
            axes[1].set_xticklabels(ticks, rotation=45, ha='right')

        handles = [
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=lulc_color(c),
                       markeredgecolor='#2d3748', markersize=8, label=f'{c} — {class_name(c)}')
            for c in sorted(clases)
        ]
        if cambiados:
            handles.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
                                      markeredgecolor='#c53030', markersize=10, label='Año que cambió'))
        leg = fig.legend(handles=handles, loc='lower center', ncol=min(4, max(1, len(handles))),
                         fontsize=8, framealpha=0.95, title='Clases LULC',
                         bbox_to_anchor=(0.5, -0.02), facecolor='white', edgecolor='#cbd5e0',
                         labelcolor='#1a202c')
        if leg is not None:
            for t in leg.get_texts():
                t.set_color('#1a202c')
            if leg.get_title() is not None:
                leg.get_title().set_color('#1a202c')
        fig.suptitle(f'Antes vs después · píxel {lat:.4f}, {lon:.4f}', fontsize=12, color='#1a202c', y=1.01)
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.18, hspace=0.28)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=110, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return buf.getvalue(), len(cambiados)


    def refrescar_serie_pixel(lat=None, lon=None, loading_text=None):
        """Consulta original y corregida del píxel marcado y actualiza el gráfico."""
        if lat is None or lon is None:
            if not state.get('pixel'):
                return
            lat, lon = state['pixel']

        # Conservar vista del mapa al actualizar el gráfico.
        vista = _capturar_vista()
        try:
            mostrar_pixel_cargando(lat, lon, loading_text or 'Consultando serie original y corregida…')
            props_o = sample_pixel_serie(IMG_ORIGINAL, lon, lat) or {}
            props_c = sample_pixel_serie(IMG_CORREGIDA, lon, lat) or {}
            df_o = props_a_serie(props_o)
            df_c = props_a_serie(props_c)
            if df_o.empty and df_c.empty:
                mostrar_pixel_resultado(lat, lon, empty=True)
                return
            png, n_cambios = render_compare_png(lat, lon, df_o, df_c)
            mostrar_pixel_resultado(lat, lon, png_bytes=png, empty=png is None, n_cambios=n_cambios)
        finally:
            _restaurar_vista_diferida(vista)


    def _modo_mapa():
        modo = str(globals().get('MAPA_ANOMALIAS', 'bosque')).strip().lower()
        if modo in ('bosque_plantacion', 'bosque+plantacion', 'plantacion', 'plantaciones') or 'plant' in modo:
            return 'bosque_plantacion'
        return 'bosque'


    def _txt_modo_mapa(modo=None):
        modo = modo or _modo_mapa()
        if modo == 'bosque_plantacion':
            return 'Bosque (rojo) + plantaciones 9 35 74 (azul)'
        return 'Solo bosque'


    def analizar_imagen(imagen, fase, refrescar_pixel=True):
        state['current_img'] = imagen
        state['fase'] = fase
        modo_mapa = _modo_mapa()
        fase_label.value = f'<span style="color:#4a5568">Vista activa: <b>{fase}</b></span>'
        vista = _capturar_vista() if state.get('vista_inicializada') else None
        quitar_capas_analisis()

        # Mantener marcador si existe.
        if state.get('pixel'):
            lat, lon = state['pixel']
            poner_marcador(lat, lon)

        capa_txt = _txt_modo_mapa(modo_mapa)
        table_html.value = (
            f"{TABLE_CSS}<div class='traj-wrap'><h3>Trayectorias sospechosas — {fase}</h3>"
            f"<div class='meta'>Mapa: {capa_txt} (MAPA_ANOMALIAS) · Calculando…</div></div>"
        )

        def mapear_anomalias(y):
            year = ee.Number(y)
            prev1 = imagen.select(band_name(year.subtract(1)))
            curr = imagen.select(band_name(year))
            next1 = imagen.select(band_name(year.add(1)))
            next2 = imagen.select(band_name(year.add(2)))
            a_bosque = ee.Image(anomalia_bosque(prev1, curr, next1, next2)).unmask(0).gt(0)
            a_plant = ee.Image(anomalia_plantacion(prev1, curr, next1, next2)).unmask(0).gt(0)
            if modo_mapa == 'bosque_plantacion':
                anomalia = a_bosque.Or(a_plant)
                trayectoria = prev1.multiply(1_000_000).add(curr.multiply(10_000)).add(next1.multiply(100)).add(next2)
                return ee.Image([
                    a_bosque.rename('error_bosque'),
                    a_plant.rename('error_plant'),
                    anomalia.rename('error'),
                    trayectoria.updateMask(anomalia).rename('trajectory'),
                ])
            trayectoria = prev1.multiply(1_000_000).add(curr.multiply(10_000)).add(next1.multiply(100)).add(next2)
            return ee.Image([
                a_bosque.rename('error'),
                trayectoria.updateMask(a_bosque).rename('trajectory'),
            ])

        errores = ee.ImageCollection.fromImages(YEARS.map(mapear_anomalias))
        state['trajectories_img'] = errores.select('trajectory').max()

        # Identificadores de capa.
        fase_tag = fase.replace(' ', '_').replace('·', '')
        layer_lulc = f'LULC2023_{fase_tag}'
        m.add_ee_layer(imagen.select(band_name(2023)), VIS_LULC, name=layer_lulc, shown=False, opacity=0.35)
        layers = [layer_lulc]

        def _freq(band):
            img = errores.select(band).sum().unmask(0)
            return img.where(img.gt(4), 4).selfMask()

        if modo_mapa == 'bosque_plantacion':
            # Capas separadas: bosque y plantación.
            layer_b = f'anom_bosque_{fase_tag}'
            layer_p = f'anom_plant_{fase_tag}'
            m.add_ee_layer(_freq('error_bosque'), VIS_ANOMALIAS, name=layer_b, shown=True, opacity=0.85)
            m.add_ee_layer(
                _freq('error_plant'),
                globals().get('VIS_ANOMALIAS_PLANT', VIS_ANOMALIAS),
                name=layer_p, shown=True, opacity=0.85,
            )
            layers.extend([layer_b, layer_p])
            legend_dict = globals().get('LEGEND_ANOMALIAS_MIXTO', LEGEND_ANOMALIAS)
            legend_title = 'Bosque (rojo) vs plantacion (azul)'
        else:
            layer_anom = f'anom_bosque_{fase_tag}'
            m.add_ee_layer(_freq('error'), VIS_ANOMALIAS, name=layer_anom, shown=True, opacity=0.85)
            layers.append(layer_anom)
            legend_dict = LEGEND_ANOMALIAS
            legend_title = 'Frecuencia bosque aislado'

        state['layers'] = layers

        if not state['legend_added']:
            m.add_legend(title=legend_title, legend_dict=legend_dict, position='bottomleft')
            state['legend_added'] = True

        # Encaja al área solo al cargar; luego no saltes al zoom general.
        if not state.get('vista_inicializada'):
            if REGION_ID is None:
                m.fit_bounds([[-4.5, -82.0], [13.5, -66.5]])
            else:
                bounds = imagen.geometry().bounds().getInfo()['coordinates'][0]
                lats = [p[1] for p in bounds]
                lons = [p[0] for p in bounds]
                m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
            state['vista_inicializada'] = True
        else:
            _restaurar_vista_diferida(vista)

        if REGION_ID is None:
            if fase != 'Original':
                table_html.value = (
                    f"{TABLE_CSS}<div class='traj-wrap'><h3>Trayectorias — {fase}</h3>"
                    "<div class='empty'>El mapa corregido nacional se visualiza, pero su tabla no se recalcula en GEE.</div></div>"
                )
            elif not OUTPUT_CSV.exists():
                table_html.value = (
                    f"{TABLE_CSS}<div class='traj-wrap'><h3>Trayectorias — Nacional</h3>"
                    f"<div class='empty'>No existe {OUTPUT_CSV.name}.</div></div>"
                )
            else:
                src = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
                keys = ['codigo', 'clase_1', 'clase_2', 'clase_3', 'clase_4', 'clases', 'trayectoria', 'tipo']
                df = (
                    src.groupby([k for k in keys if k in src.columns], as_index=False)['pixeles']
                    .sum().sort_values('pixeles', ascending=False)
                )
                table_html.value = html_tabla_trayectorias(df, 'Original · nacional')
        else:
            hist = state['trajectories_img'].reduceRegion(
                reducer=ee.Reducer.frequencyHistogram(),
                geometry=imagen.geometry(), scale=30, maxPixels=1e10, bestEffort=True,
            ).getInfo()
            if not hist.get('trajectory'):
                table_html.value = (
                    f"{TABLE_CSS}<div class='traj-wrap'><h3>Trayectorias — {fase}</h3>"
                    f"<div class='empty'>Sin anomalías detectadas.</div></div>"
                )
            else:
                df = pd.DataFrame([
                    {**parse_trajectory(k), 'pixeles': v} for k, v in hist['trajectory'].items()
                ]).sort_values('pixeles', ascending=False)
                table_html.value = html_tabla_trayectorias(df, f'{fase} · {_txt_modo_mapa(modo_mapa)}')

        # Si había un píxel marcado, refresca la comparación sin perderlo ni la vista.
        if refrescar_pixel and state.get('pixel'):
            lat, lon = state['pixel']
            poner_marcador(lat, lon)
            refrescar_serie_pixel(lat, lon, loading_text=f'Actualizando comparación ({fase})…')

        if vista is not None:
            _restaurar_vista_diferida(vista)


    def on_map_click(**kwargs):
        if state.get('map_seq') != _MAP_UI.get('seq'):
            return
        if kwargs.get('type') != 'click':
            return
        if _MAP_UI.get('busy'):
            return

        now = time.time()
        if now - state['last_click_ts'] < 0.6:
            return
        state['last_click_ts'] = now
        _MAP_UI['busy'] = True
        vista = _capturar_vista()

        try:
            lat, lon = kwargs['coordinates']
            state['pixel'] = (lat, lon)
            poner_marcador(lat, lon)
            refrescar_serie_pixel(lat, lon)
        except Exception as err:
            try:
                lat, lon = kwargs.get('coordinates', (0, 0))
            except Exception:
                lat, lon = 0, 0
            mostrar_pixel_resultado(lat, lon, error=err)
        finally:
            _restaurar_vista_diferida(vista)
            _MAP_UI['busy'] = False


    m.on_interaction(on_map_click)

    btn_original = widgets.Button(description=' Ver original', button_style='info', icon='map', layout=widgets.Layout(width='140px'))
    btn_corregida = widgets.Button(description=' Ver matriz corregida', button_style='warning', icon='wrench', layout=widgets.Layout(width='190px'))

    def _analizar(btn, imagen, fase):
        btn.disabled = True
        try:
            analizar_imagen(imagen, fase)
        finally:
            btn.disabled = False

    btn_original.on_click(lambda b: _analizar(b, IMG_ORIGINAL, 'Original'))
    btn_corregida.on_click(lambda b: _analizar(b, IMG_CORREGIDA, 'Corregida · matriz'))

    ambito_mapa = f'Región {REGION_ID}' if REGION_ID is not None else 'Todas las regiones disponibles'
    header = widgets.VBox([
        widgets.HTML(
            '<div style="font-family:system-ui;font-size:15px;margin:4px 0 10px 0">'
            f'<b>{ambito_mapa}</b> · Anomalías: <b>{_txt_modo_mapa()}</b> '
            f'(config MAPA_ANOMALIAS) · Marca un píxel para comparar.'
            '</div>'
        ),
        widgets.HBox([btn_original, btn_corregida, fase_label], layout=widgets.Layout(align_items='center', gap='12px')),
    ])

    panels = widgets.VBox([
        _section_title('Mapa', ambito_mapa),
        m,
        _section_title('Serie del píxel · original vs corregida'),
        chart_box,
        _section_title('Patrones', ambito_mapa),
        table_html,
    ], layout=widgets.Layout(width='100%'))

    clear_output(wait=True)
    display(widgets.VBox([header, panels], layout=widgets.Layout(width='100%')))
    mostrar_placeholder_pixel()
    analizar_imagen(IMG_ORIGINAL, 'Original')


## 9. Publicar en Earth Engine


In [ ]:
if not MODO_MAPA:
    print('Define REGION_ID y ejecuta la celda de carga antes de exportar.')
else:
    VERSION_OUT = '99'
    asset_id = f'{FOLDER}/COLOMBIA-{REGION_ID}-{VERSION_OUT}_99'

    task = ee.batch.Export.image.toAsset(
        # CORRECCIÓN: Envolver la operación con ee.Image()
        image=ee.Image(IMG_CORREGIDA.copyProperties(IMG_ORIGINAL, IMG_ORIGINAL.propertyNames())),
        description=f'COLOMBIA-{REGION_ID}-{VERSION_OUT}',
        assetId=asset_id,
        pyramidingPolicy={'.default': 'mode'},
        region=IMG_ORIGINAL.geometry().bounds(),
        scale=30,
        maxPixels=1e13,
    )
    task.start()
    print(f'Tarea iniciada: {asset_id}')